# Transform — Pipeline Otimizado para Clusterização
**Projeto:** Análise de padrões de hospitalização em casos de dengue — SINAN/DATASUS

Responsável: Davi de Souza Lopes

## 1. Dependências e configuração


In [1]:
!pip install -q pyspark pandas pyarrow

In [2]:
from google.colab import drive
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, trim, upper, sum as spark_sum
)
from pyspark.sql import functions as F
import pandas as pd

In [3]:
drive.mount('/content/drive')

base_path      = '/content/drive/MyDrive/Topicos_BD'
raw_sinan      = os.path.join(base_path, 'raw', 'sinan', 'DENGBR*.parquet')
processed_path = os.path.join(base_path, 'processed')
os.makedirs(processed_path, exist_ok=True)

print('Caminhos configurados:')
print(f'  raw/sinan  → {raw_sinan}')
print(f'  processed  → {processed_path}')

Mounted at /content/drive
Caminhos configurados:
  raw/sinan  → /content/drive/MyDrive/Topicos_BD/raw/sinan/DENGBR*.parquet
  processed  → /content/drive/MyDrive/Topicos_BD/processed


In [4]:
spark = SparkSession.builder \
    .appName('DataMajor_Transform_Otimizado') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

print(f'Spark versão: {spark.version}')

Spark versão: 4.0.2


## 2. Variáveis selecionadas

Selecionamos **apenas as colunas que serão usadas** antes de qualquer transformação para reduzir o volume de dados em memória

In [5]:
COLUNAS_INTERESSE = [
    # Desfechos (usados para interpretação pós-cluster)
    'HOSPITALIZ', 'CLASSI_FIN', 'EVOLUCAO',
    # Perfil do paciente
    'CS_SEXO', 'NU_IDADE_N',
    # Geográfico (apenas para REGIAO)
    'ID_MUNICIP',
    # Sintomas clínicos
    'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA',
    'DOR_RETRO', 'EXANTEMA', 'LEUCOPENIA',
    # Comorbidades
    'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG',
]

df = spark.read.parquet(raw_sinan).select(COLUNAS_INTERESSE)

print(f'Registros lidos: {df.count():,}')

Registros lidos: 9,615,975


## 3. Variável alvo — HOSPITALIZ

| Valor | Significado        | Ação                          |
|-------|--------------------|-------------------------------|
| `1`   | Hospitalizado      | Manter → label **1**          |
| `2`   | Não hospitalizado  | Manter → label **0**          |
| `''`  | Vazio              | Imputar via `CLASSI_FIN`      |
| `9`   | Ignorado           | **Remover**                   |
| `0`   | Inválido           | **Remover**                   |

Vazios são imputados como `1` se `CLASSI_FIN ∈ {11, 12}` (dengue grave / sinais de alarme). Os demais vazios são removidos.

> `HOSPITALIZ` **não entra nas features de clusterização** — é um desfecho. Será salva em `df_desfecho` para interpretar os clusters após a mineração.


In [6]:
# Remove inválidos
df = df.filter(~col('HOSPITALIZ').isin(['9', '0']))

# Imputa vazios via CLASSI_FIN
df = df.withColumn(
    'HOSPITALIZ',
    when(
        (trim(col('HOSPITALIZ')) == '') & col('CLASSI_FIN').isin(['11', '12']),
        '1'
    ).otherwise(col('HOSPITALIZ'))
)
df = df.filter(trim(col('HOSPITALIZ')) != '')

# Converte para binário
df = df.withColumn(
    'HOSPITALIZ',
    when(col('HOSPITALIZ') == '1', 1)
    .when(col('HOSPITALIZ') == '2', 0)
    .otherwise(None)
).filter(col('HOSPITALIZ').isNotNull())

# Remove casos descartados de dengue
df = df.filter(col('CLASSI_FIN') != '5')

print(f'Registros após filtros de HOSPITALIZ e CLASSI_FIN: {df.count():,}')

Registros após filtros de HOSPITALIZ e CLASSI_FIN: 6,901,591


## 4. Cache

Os filtros de `HOSPITALIZ` e `CLASSI_FIN = 5` são as maiores reduções de volume. Cachear aqui evita re-leitura do parquet em todas as transformações seguintes.


In [7]:
df = df.cache()
total_cached = df.count()  # força materialização do cache
print(f'DataFrame cacheado: {total_cached:,} registros')

DataFrame cacheado: 6,901,591 registros


## 5. Conversão de tipos dos desfechos

`CLASSI_FIN` e `EVOLUCAO` precisam de cast antes de serem separadas.


In [8]:
# EVOLUCAO: 1=Cura, 2=Óbito por dengue, 3=Óbito por outra causa, 9=Ignorado
df = df.withColumn(
    'EVOLUCAO',
    when(col('EVOLUCAO').isin(['9']) | (trim(col('EVOLUCAO')) == ''), None)
    .otherwise(col('EVOLUCAO').cast('int'))
)

# CLASSI_FIN: 10=Dengue, 11=Dengue c/ sinais de alarme, 12=Dengue grave
df = df.withColumn(
    'CLASSI_FIN',
    when(trim(col('CLASSI_FIN')) == '', None)
    .otherwise(col('CLASSI_FIN').cast('int'))
)

## 6. Separação das variáveis de desfecho

| Variável      | Papel                   | Entra no K-Means? |
|---------------|-------------------------|-------------------|
| `HOSPITALIZ`  | Desfecho principal      | **NÃO**           |
| `CLASSI_FIN`  | Classificação clínica   | **NÃO**           |
| `EVOLUCAO`    | Desfecho vital          | **NÃO**           |

Essas três variáveis são salvas em `df_desfecho` e usadas **apenas para interpretar** os clusters

In [9]:
# Separação das variáveis de desfecho
df = df.withColumn('_row_id', F.monotonically_increasing_id())

df_desfecho = df.select('_row_id', 'HOSPITALIZ', 'CLASSI_FIN', 'EVOLUCAO')
df = df.drop('HOSPITALIZ', 'CLASSI_FIN', 'EVOLUCAO')

print(f'Desfechos separados — {len(df.columns)} colunas restantes em df.')

Desfechos separados — 17 colunas restantes em df.


## 7. Enriquecimento geográfico — variáveis REGIAO (do IBGE)

Join SINAN x IBGE por `ID_MUNICIP` para obter a região geográfica (`Norte`, `Nordeste`, `Centro-Oeste`, `Sudeste`, `Sul`).

**Obs.: — `REGIAO` é apenas para análise pós-cluster:**
- Adicionada a `df_desfecho` (junto com `HOSPITALIZ`, `CLASSI_FIN`, `EVOLUCAO`)
- **NÃO** entra em `FEATURE_COLS`
- **NÃO** recebe `OneHotEncoder` / `StringIndexer`
- **NÃO** participa do `VectorAssembler` → PCA → BisectingKMeans

Após o join, `ID_MUNICIP`, `nome_municipio`, `uf_sigla`, `uf_nome` e colunas auxiliares do IBGE são descartadas — apenas `REGIAO` é preservada.

In [10]:
# Join IBGE para obtenção da região
raw_ibge = os.path.join(base_path, 'raw', 'ibge', 'ibge_municipios.parquet')
df_ibge = pd.read_parquet(raw_ibge)
df_ibge_spark = spark.createDataFrame(df_ibge)

df = df.join(
    df_ibge_spark.select('id_municipio_sinan', 'regiao'),
    df.ID_MUNICIP == df_ibge_spark.id_municipio_sinan,
    how='left'
)

df = df.withColumnRenamed('regiao', 'REGIAO')
df = df.withColumn(
    'REGIAO',
    when(col('REGIAO').isNull(), 'Sem_Regiao').otherwise(col('REGIAO'))
)

df = df.drop('ID_MUNICIP', 'id_municipio_sinan')

print('REGIAO criada com sucesso.')

REGIAO criada com sucesso.


In [11]:
# REGIAO disponível em df_desfecho para análise pós-cluster
df_desfecho = df_desfecho.join(
    df.select('_row_id', 'REGIAO'),
    on='_row_id', how='left'
)

# REGIAO não participa da clusterização — mantida apenas em df_desfecho
df = df.drop('REGIAO')

## 8. Sintomas e comorbidades — padronização 1/0

No SINAN: `1 = Sim`, `2 = Não`, `9 = Ignorado` (tratado como ausente).

**Regra:** `9` e vazios → `None` (não são convertidos para `0` = "Não"). A imputação é feita pela **moda real** na etapa seguinte.

> `DOR_COSTAS` foi removida da lista — não é campo padrão do dicionário SINAN dengue. Verificar se existe na sua base antes de reincluir.


In [12]:
COLUNAS_BINARIAS = [
    # Sintomas clínicos
    'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA',
    'DOR_RETRO', 'EXANTEMA', 'LEUCOPENIA',
    # Comorbidades
    'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG',
]

for c in COLUNAS_BINARIAS:
    if c in df.columns:
        df = df.withColumn(
            c,
            when(col(c) == '1', 1)
            .when(col(c) == '2', 0)
            .otherwise(None)  # '9', vazio, nulo → None
        )

## 9. Idade — `NU_IDADE_N` → `idade_anos`

O SINAN codifica: `1XXX` = anos, `2XXX` = meses, `3XXX` = dias.
Apenas registros em anos são mantidos como valor real. Recém-nascidos e casos em meses/dias recebem `idade_anos = 0`.


In [13]:
df = df.withColumn(
    'idade_anos',
    when(
        col('NU_IDADE_N').startswith('1'),
        col('NU_IDADE_N').substr(2, 3).cast('int')
    ).otherwise(0)
).filter(
    col('idade_anos').isNotNull() &
    (col('idade_anos') >= 0) &
    (col('idade_anos') <= 120)
).drop('NU_IDADE_N')

## 10. Sexo — `CS_SEXO` → `is_mulher`

**Dummy:** mantemos **apenas `is_mulher`** (1=F, 0=M). Manter `is_homem` junto seria colinearidade perfeita — o PCA criaria um componente artificial para essa relação.

`is_sexo_ausente` foi removida pois a prevalência de CS_SEXO ausente era de apenas < 0,5%


In [14]:
df = df.withColumn('CS_SEXO', trim(upper(col('CS_SEXO'))))

# Apenas is_mulher — evita dummy trap
df = df.withColumn(
    'is_mulher',
    when(col('CS_SEXO') == 'F', 1)
    .when(col('CS_SEXO') == 'M', 0)
    .otherwise(None)  # outros → nulo (imputar na próxima etapa)
).drop('CS_SEXO')

## 11. Tratamento de nulos

**Etapa 1 — Remover colunas com > 30% de nulos** (dados insuficientes para inferência).
**Etapa 2 — Imputar binárias e ordinais pela moda/mediana** (preserva distribuição real).
**Etapa 3 — Remover linhas com nulos em `idade_anos` e `is_mulher`** (essenciais).

> A imputação por moda usa `agg(sum)`


In [15]:
# Remove colunas com alta nulidade (> 30%)
total = df.count()

COLUNAS_BLINDADAS = ['is_mulher', 'idade_anos']

nulos_expr = {c: count(when(col(c).isNull(), c)).alias(c) for c in df.columns}
nulos_row  = df.agg(*nulos_expr.values()).collect()[0].asDict()

colunas_alta_nulidade = [
    c for c, n in nulos_row.items()
    if (n / total * 100) > 30 and c not in COLUNAS_BLINDADAS
]

if colunas_alta_nulidade:
    print(f'Removidas por alta nulidade (> 30%): {colunas_alta_nulidade}')
    df = df.drop(*colunas_alta_nulidade)
else:
    print('Nenhuma coluna com > 30% de nulos.')

Nenhuma coluna com > 30% de nulos.


In [16]:
# Imputação de nulos pela moda (binárias e is_mulher)
colunas_para_impute = [
    c for c in df.columns
    if c in COLUNAS_BINARIAS + ['is_mulher']
    and c in df.columns
]

# sum(col) / count(*) > 0.5  → moda = 1; caso contrário moda = 0
agg_exprs = [
    F.sum(when(col(c) == 1, 1).otherwise(0)).alias(f'sum_{c}')
    for c in colunas_para_impute
]
agg_result = df.agg(*agg_exprs, F.count('*').alias('total')).collect()[0].asDict()
n_total = agg_result['total']

modas = {
    c: (1 if agg_result[f'sum_{c}'] / n_total > 0.5 else 0)
    for c in colunas_para_impute
}

df = df.fillna(modas)

In [17]:
# Remove linhas com nulos em colunas essenciais
antes = df.count()
df = df.dropna(subset=['idade_anos', 'is_mulher'])
depois = df.count()
print(f'Registros finais: {depois:,} (removidos: {antes - depois:,})')

Registros finais: 6,901,591 (removidos: 0)


## 12. Seleção final de features

**Features que entram no StandardScaler → PCA → Bisecting K-Means:**

| Grupo          | Variáveis                                            |
|----------------|------------------------------------------------------|
| Perfil social  | `is_mulher`                                          |
| Idade          | `idade_anos`                                         |
| Sintomas       | `FEBRE`, `MIALGIA`, `CEFALEIA`, `VOMITO`, `NAUSEA`, `DOR_RETRO`, `EXANTEMA`, `LEUCOPENIA` |
| Comorbidades   | `DIABETES`, `HIPERTENSA`, `RENAL`, `HEPATOPAT`, `HEMATOLOG` |

### Validação de compatibilidade com Spark ML

| Feature                   | Tipo | % ausência | Estratégia                  | None no parquet |
|---------------------------|------|------------|-----------------------------|-----------------|
| `idade_anos`              | int  | ~0%        | calculada diretamente       | 0               |
| `is_mulher`               | int  | <0,5%      | imputada pela moda          | 0               |
| `FEBRE`…`HEMATOLOG`       | int  | variável   | imputadas pela moda         | 0               |

> Todas as features são `int` sem `None` — compatíveis com `VectorAssembler` sem nenhum descarte de registros.

**Excluídas das features (e por quê):**

| Variável         | Motivo da exclusão                                        |
|------------------|-----------------------------------------------------------|
| `HOSPITALIZ`     | Desfecho — invalida a clusterização se incluída           |
| `CLASSI_FIN`     | Desfecho clínico posterior ao atendimento                 |
| `EVOLUCAO`       | Desfecho vital posterior ao atendimento                   |
| `is_homem`       | Colinear com `is_mulher` (dummy trap)                     |
| `is_sexo_ausente`| Quase constante — adiciona ruído ao PCA                   |
| `_row_id`        | Índice interno de recruzamento                            |
| `REGIAO`         | Geográfica — mantida apenas em `df_desfecho` para análise pós-cluster |
| `CS_ESCOL_N`     | Removida — elevada proporção de ausentes/inconsistências, evita viés analítico |
| Colunas de raça  | V3 — não usa variáveis de raça                           |


In [18]:
# Colunas excluídas das features de mineração
EXCLUIR = [
    '_row_id',
    # Escolaridade — removida (alta ausência/inconsistência)
    'CS_ESCOL_N', 'escolaridade_ord', 'is_escolaridade_ausente',
    # Geográficas — REGIAO mantida apenas em df_desfecho
    'REGIAO', 'regiao', 'uf_sigla', 'uf_nome', 'nome_municipio', 'ID_MUNICIP',
    # Raça — não utilizada nesta versão
    'is_branco', 'is_preto', 'is_pardo', 'is_amarelo', 'is_indigena', 'is_raca_ausente',
]

EXCLUIR_PRESENTES = [c for c in EXCLUIR if c in df.columns]
df_mineracao = df.drop(*EXCLUIR_PRESENTES)

FEATURE_COLS = [
    'idade_anos',
    'is_mulher',

    'FEBRE',
    'MIALGIA',
    'CEFALEIA',
    'VOMITO',
    'NAUSEA',
    'DOR_RETRO',
    'EXANTEMA',
    'LEUCOPENIA',

    'DIABETES',
    'HIPERTENSA',
    'RENAL',
    'HEPATOPAT',
    'HEMATOLOG'
]

# Confere integridade do conjunto final de features
faltando = [c for c in FEATURE_COLS if c not in df_mineracao.columns]
extras   = [c for c in df_mineracao.columns if c not in FEATURE_COLS]
if faltando:
    raise ValueError(f'FEATURE_COLS espera colunas ausentes em df_mineracao: {faltando}')
if extras:
    print(f'AVISO — colunas extras não listadas em FEATURE_COLS: {extras}')

print(f'Dataset de mineração: {df_mineracao.count():,} registros, {len(df_mineracao.columns)} features')

Dataset de mineração: 6,901,591 registros, 15 features


## 13. Relatório final


In [19]:
total_final    = df_mineracao.count()
total_original = 6_102_102

print()
print('Resumo Transform — V3 Otimizado (sem escolaridade, sem raça, sem HOSPITALIZ nas features)')
print('=' * 65)
print(f'Registros originais:          {total_original:>12,}')
print(f'Registros após Transform:     {total_final:>12,}')
print(f'Registros removidos:          {total_original - total_final:>12,}')
print(f'Retenção:                     {total_final/total_original*100:>11.1f}%')
print()
print('Transformações aplicadas:')
print('  [OK] HOSPITALIZ: 9 e 0 removidos; vazios imputados via CLASSI_FIN')
print('  [OK] CLASSI_FIN = 5 (descartados) removidos')
print('  [OK] Binárias: 2→0, 1→1, 9→None; nulos → moda (via agg único)')
print('  [OK] NU_IDADE_N → idade_anos (0–120 anos)')
print('  [OK] CS_ESCOL_N removida — elevada ausência/inconsistência, evita viés analítico')
print('  [OK] CS_SEXO → is_mulher (dummy trap corrigida)')
print('  [OK] HOSPITALIZ, CLASSI_FIN, EVOLUCAO → df_desfecho')
print('  [OK] Join IBGE eliminado (zero impacto na V3)')
print('  [OK] cache() após filtros críticos')
print()
print(f'  Features de mineração: {len(df_mineracao.columns)} colunas')
print('=' * 65)


Resumo Transform — V3 Otimizado (sem escolaridade, sem raça, sem HOSPITALIZ nas features)
Registros originais:             6,102,102
Registros após Transform:        6,901,591
Registros removidos:              -799,489
Retenção:                           113.1%

Transformações aplicadas:
  [OK] HOSPITALIZ: 9 e 0 removidos; vazios imputados via CLASSI_FIN
  [OK] CLASSI_FIN = 5 (descartados) removidos
  [OK] Binárias: 2→0, 1→1, 9→None; nulos → moda (via agg único)
  [OK] NU_IDADE_N → idade_anos (0–120 anos)
  [OK] CS_ESCOL_N removida — elevada ausência/inconsistência, evita viés analítico
  [OK] CS_SEXO → is_mulher (dummy trap corrigida)
  [OK] HOSPITALIZ, CLASSI_FIN, EVOLUCAO → df_desfecho
  [OK] Join IBGE eliminado (zero impacto na V3)
  [OK] cache() após filtros críticos

  Features de mineração: 15 colunas


## 14. Salvar em `processed/`


In [20]:
output_mineracao = os.path.join(processed_path, 'sinan_dengue_features_v3')
output_desfecho  = os.path.join(processed_path, 'sinan_dengue_desfecho_v3')

df_mineracao.write.mode('overwrite').parquet(output_mineracao)
df_desfecho.write.mode('overwrite').parquet(output_desfecho)

print('Dataset salvo com sucesso:')
print(f'  features  → {output_mineracao}')
print(f'  desfecho  → {output_desfecho}')

spark.stop()
print('Transform concluído.')

Dataset salvo com sucesso:
  features  → /content/drive/MyDrive/Topicos_BD/processed/sinan_dengue_features_v3
  desfecho  → /content/drive/MyDrive/Topicos_BD/processed/sinan_dengue_desfecho_v3
Transform concluído.
